# Home Consumables Tracker
Creates/updates a Google Sheet of replaceable household items, checks prices,
and pushes a phone notification (via ntfy.sh) when something is due.

**One-time setup**
1. Install the free **ntfy** app on your phone, subscribe to a long random topic name (e.g. `asa-home-x7k29q`).
2. In Colab: key icon (Secrets) -> add `NTFY_TOPIC` = that topic name -> toggle "Notebook access" on.
3. Run all cells. First run creates the sheet "Home Consumables Tracker" in your Drive and seeds it.
4. In the sheet: fill in **Product URL** and **Last Replaced** for each item you actually own. Delete the rest.

**Day to day:** re-run the last cell whenever you want a check. After you swap a filter, set its
*Last Replaced* date in the sheet (or run `mark_replaced("Item name")`).

In [ ]:
# Config
LEAD_DAYS = 14          # alert this many days before the due date
RENOTIFY_DAYS = 3       # don't re-alert on the same item more often than this
CHECK_PRICES = True     # try to scrape prices from the Product URL column
SHEET_NAME = "Home Consumables Tracker"
ITEMS_TAB = "Items"
HISTORY_TAB = "Price History"

In [ ]:
# Imports and auth
import re, json, time, random, smtplib, datetime as dt
from email.message import EmailMessage

import requests
import gspread
from bs4 import BeautifulSoup
from dateutil import parser as dtparse
from google.colab import auth, userdata
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)


def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None


NTFY_TOPIC = secret("NTFY_TOPIC")
GMAIL_ADDRESS = secret("GMAIL_ADDRESS")            # optional
GMAIL_APP_PASSWORD = secret("GMAIL_APP_PASSWORD")  # optional (Google account app password)

In [ ]:
# Sheet layout and starter items
HEADERS = [
    "Item", "Location", "Category", "Product Name / Model", "Product URL",
    "Replace Every (days)", "Last Replaced", "Next Due", "Days Until Due", "Status",
    "Current Price", "Lowest Price Seen", "Price Checked", "Last Notified", "Notes",
]
COL = {k: i for i, k in enumerate(
    ["item", "loc", "cat", "product", "url", "every", "last", "next", "days",
     "status", "price", "low", "checked", "notified", "notes"])}

# (item, location, category, interval_days, note) -- intervals are starting points, tune to your home
SEED = [
    ("Cat water fountain filter", "Kitchen", "Pets", 30, "Check your fountain's manual; many say 2-4 weeks"),
    ("Air purifier filter", "Living room", "Air quality", 180, "HEPA/carbon; shorter with pets or smoke"),
    ("HVAC filter - upstairs", "Upstairs", "Air quality", 90, "Note the size printed on the old filter"),
    ("HVAC filter - downstairs", "Downstairs", "Air quality", 90, "Note the size printed on the old filter"),
    ("Fridge water filter", "Kitchen", "Water", 180, ""),
    ("Water pitcher filter", "Kitchen", "Water", 60, ""),
    ("Whole-house sediment filter", "Basement/garage", "Water", 120, ""),
    ("Showerhead filter cartridge", "Bathroom", "Water", 90, ""),
    ("Range hood filter", "Kitchen", "Appliance", 90, "Charcoal type; metal mesh types get washed instead"),
    ("Vacuum filter", "Closet", "Cleaning", 180, ""),
    ("Robot vacuum brush + filter", "Living room", "Cleaning", 90, ""),
    ("Humidifier filter / wick", "Bedroom", "Air quality", 90, ""),
    ("Dishwasher cleaner tablet", "Kitchen", "Appliance", 30, ""),
    ("Washing machine cleaner", "Laundry", "Appliance", 90, ""),
    ("Dryer vent cleaning", "Laundry", "Safety", 365, "Fire prevention"),
    ("Smoke detector batteries", "Whole home", "Safety", 365, ""),
    ("Smoke detector units", "Whole home", "Safety", 3650, "Replace whole unit every ~10 years"),
    ("CO detector units", "Whole home", "Safety", 2190, "Check the date on the back; typically 5-7 years"),
    ("Fire extinguisher (replace/service)", "Kitchen", "Safety", 4380, "Check gauge yearly; replace per label"),
    ("Electric toothbrush heads", "Bathroom", "Personal", 90, ""),
    ("Kitchen sponges / scrubbers", "Kitchen", "Cleaning", 21, ""),
    ("Smart lock batteries", "Front door", "Safety", 180, ""),
    ("Toilet flapper", "Bathroom", "Plumbing", 730, "Cheap fix that stops silent leaks"),
    ("Water heater flush", "Basement/garage", "Plumbing", 365, "Service, not a purchase; still worth tracking"),
]

HIST_HEADERS = ["Date", "Item", "Price", "URL"]
MAX_ROWS = 150  # formulas are pre-filled this far down so new rows just work


def status_formula(r):
    return (f'=IF($A{r}="","",IF($G{r}="","SET DATE",'
            f'IF($I{r}<0,"OVERDUE",IF($I{r}<={LEAD_DAYS},"DUE SOON","OK"))))')


def col_a1(row, key):
    return gspread.utils.rowcol_to_a1(row, COL[key] + 1)

In [ ]:
# Sheet setup
def setup(seed=True):
    try:
        sh = gc.open(SHEET_NAME)
    except gspread.SpreadsheetNotFound:
        sh = gc.create(SHEET_NAME)
        print("Created sheet:", sh.url)

    try:
        ws = sh.worksheet(ITEMS_TAB)
    except gspread.WorksheetNotFound:
        first = sh.sheet1
        if first.title.startswith("Sheet"):
            first.update_title(ITEMS_TAB)
            ws = first
        else:
            ws = sh.add_worksheet(ITEMS_TAB, MAX_ROWS + 1, len(HEADERS))
    try:
        hist = sh.worksheet(HISTORY_TAB)
    except gspread.WorksheetNotFound:
        hist = sh.add_worksheet(HISTORY_TAB, 2000, len(HIST_HEADERS))
        hist.update(values=[HIST_HEADERS], range_name="A1")
        hist.freeze(rows=1)
        hist.format("A1:D1", {"textFormat": {"bold": True}})

    if ws.row_values(1) != HEADERS:
        ws.update(values=[HEADERS], range_name="A1")
        _format_items(sh, ws)

        # live formulas for Next Due / Days Until Due / Status
        formulas = []
        for r in range(2, MAX_ROWS + 2):
            formulas.append([
                f'=IF(OR($A{r}="",$G{r}="",$F{r}=""),"",$G{r}+$F{r})',
                f'=IF($H{r}="","",$H{r}-TODAY())',
                status_formula(r),
            ])
        ws.update(values=formulas, range_name="H2", value_input_option="USER_ENTERED")

        if seed:
            rows = [[it, loc, cat, "", "", days, "", "", "", "", "", "", "", "", note]
                    for it, loc, cat, days, note in SEED]
            # write only the input columns so we don't overwrite formulas
            ws.update(values=[r[0:7] for r in rows], range_name="A2")
            ws.update(values=[[r[10], r[11], r[12], r[13], r[14]] for r in rows], range_name="K2")
        print("Sheet initialised:", sh.url)
    return sh, ws, hist


def _format_items(sh, ws):
    sid = ws.id
    reqs = [
        {"updateSheetProperties": {"properties": {"sheetId": sid, "gridProperties": {"frozenRowCount": 1}},
                                   "fields": "gridProperties.frozenRowCount"}},
        {"repeatCell": {"range": {"sheetId": sid, "startRowIndex": 0, "endRowIndex": 1},
                        "cell": {"userEnteredFormat": {"textFormat": {"bold": True},
                                                       "backgroundColor": {"red": .9, "green": .9, "blue": .9}}},
                        "fields": "userEnteredFormat(textFormat,backgroundColor)"}},
    ]
    # date + currency formats
    for key, ftype, pattern in [("last", "DATE", "yyyy-mm-dd"), ("next", "DATE", "yyyy-mm-dd"),
                                ("checked", "DATE", "yyyy-mm-dd"), ("notified", "DATE", "yyyy-mm-dd"),
                                ("price", "CURRENCY", "$#,##0.00"), ("low", "CURRENCY", "$#,##0.00")]:
        c = COL[key]
        reqs.append({"repeatCell": {
            "range": {"sheetId": sid, "startRowIndex": 1, "startColumnIndex": c, "endColumnIndex": c + 1},
            "cell": {"userEnteredFormat": {"numberFormat": {"type": ftype, "pattern": pattern}}},
            "fields": "userEnteredFormat.numberFormat"}})
    # status colours
    colours = {"OVERDUE": (.96, .72, .72), "DUE SOON": (1, .9, .6), "OK": (.72, .9, .72), "SET DATE": (.85, .85, .85)}
    for i, (text, (r, g, b)) in enumerate(colours.items()):
        reqs.append({"addConditionalFormatRule": {"index": i, "rule": {
            "ranges": [{"sheetId": sid, "startRowIndex": 1, "startColumnIndex": COL["status"],
                        "endColumnIndex": COL["status"] + 1}],
            "booleanRule": {"condition": {"type": "TEXT_EQ", "values": [{"userEnteredValue": text}]},
                            "format": {"backgroundColor": {"red": r, "green": g, "blue": b}}}}}})
    sh.batch_update({"requests": reqs})

In [ ]:
# Price fetching (best effort)
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36",
      "Accept-Language": "en-US,en;q=0.9"}


def _num(s):
    m = re.search(r"\d[\d,]*\.?\d*", str(s))
    return float(m.group().replace(",", "")) if m else None


def fetch_price(url):
    """Returns a float or None. Many retailers (Amazon especially) block scripts, so None is normal."""
    try:
        r = requests.get(url, headers=UA, timeout=20)
    except requests.RequestException:
        return None
    if r.status_code != 200:
        return None
    soup = BeautifulSoup(r.text, "html.parser")

    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
        except Exception:
            continue
        for node in (data if isinstance(data, list) else [data]):
            if not isinstance(node, dict):
                continue
            for n in node.get("@graph", [node]):
                offers = n.get("offers") if isinstance(n, dict) else None
                if isinstance(offers, list):
                    offers = offers[0] if offers else None
                if isinstance(offers, dict):
                    p = _num(offers.get("price") or offers.get("lowPrice") or "")
                    if p:
                        return p

    for name, attrs in [("meta", {"property": "product:price:amount"}),
                        ("meta", {"property": "og:price:amount"}),
                        ("meta", {"itemprop": "price"})]:
        t = soup.find(name, attrs)
        if t and t.get("content"):
            p = _num(t["content"])
            if p:
                return p

    t = soup.select_one(".a-price .a-offscreen")  # Amazon
    if t:
        return _num(t.get_text())
    return None

In [ ]:
# Notifications
def send_ntfy(title, body, urgent=False):
    if not NTFY_TOPIC:
        print("NTFY_TOPIC secret not set - printing instead:\n", title, "\n", body)
        return
    requests.post(f"https://ntfy.sh/{NTFY_TOPIC}", data=body.encode("utf-8"),
                  headers={"Title": title, "Priority": "high" if urgent else "default", "Tags": "house,bell"},
                  timeout=15)


def send_email(subject, body):
    if not (GMAIL_ADDRESS and GMAIL_APP_PASSWORD):
        return
    msg = EmailMessage()
    msg["Subject"], msg["From"], msg["To"] = subject, GMAIL_ADDRESS, GMAIL_ADDRESS
    msg.set_content(body)
    with smtplib.SMTP_SSL("smtp.gmail.com", 465) as s:
        s.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        s.send_message(msg)

In [ ]:
# Main run
def _date(s):
    try:
        return dtparse.parse(str(s)).date() if str(s).strip() else None
    except Exception:
        return None


def run(check_prices=CHECK_PRICES, notify=True):
    sh, ws, hist = setup()
    rows = ws.get_all_values()
    today = dt.date.today()
    cells, hist_rows, alerts, unset, price_failed = [], [], [], [], []

    for r_i, row in enumerate(rows[1:], start=2):
        row = row + [""] * (len(HEADERS) - len(row))
        item = row[COL["item"]].strip()
        if not item:
            continue
        url = row[COL["url"]].strip()
        every = _num(row[COL["every"]]) or None
        last = _date(row[COL["last"]])

        # --- price check
        price = _num(row[COL["price"]]) if row[COL["price"]] else None
        low = _num(row[COL["low"]]) if row[COL["low"]] else None
        if check_prices and url.startswith("http"):
            p = fetch_price(url)
            time.sleep(random.uniform(1, 3))
            if p:
                price = p
                low = min(low, p) if low else p
                cells += [{"range": col_a1(r_i, "price"), "values": [[p]]},
                          {"range": col_a1(r_i, "low"), "values": [[low]]},
                          {"range": col_a1(r_i, "checked"), "values": [[today.isoformat()]]}]
                hist_rows.append([today.isoformat(), item, p, url])
            else:
                price_failed.append(item)

        # --- due check
        if not (last and every):
            unset.append(item)
            continue
        due = last + dt.timedelta(days=int(every))
        left = (due - today).days
        if left > LEAD_DAYS:
            continue
        ln = _date(row[COL["notified"]])
        if ln and (today - ln).days < RENOTIFY_DAYS:
            continue

        tag = f"OVERDUE by {-left}d" if left < 0 else ("due TODAY" if left == 0 else f"due in {left}d")
        line = f"{'🔴' if left < 0 else '🟡'} {item} - {tag}"
        if price:
            line += f" | ${price:,.2f}"
            if low and price <= low:
                line += " (lowest seen - good time to buy)"
        if url:
            line += f"\n   {url}"
        alerts.append((left, line))
        cells.append({"range": col_a1(r_i, "notified"), "values": [[today.isoformat()]]})

    if cells:
        ws.batch_update(cells, value_input_option="USER_ENTERED")
    if hist_rows:
        hist.append_rows(hist_rows, value_input_option="USER_ENTERED")

    print(f"Checked {len(rows) - 1} rows | alerts: {len(alerts)} | no date set: {len(unset)} | "
          f"price lookups failed: {len(price_failed)}")
    if price_failed:
        print("  Price scrape failed (type the price in manually if you care):", ", ".join(price_failed))

    if alerts and notify:
        alerts.sort()
        body = "\n".join(l for _, l in alerts)
        urgent = alerts[0][0] < 0
        title = f"{len(alerts)} home item(s) need action"
        send_ntfy(title, body, urgent)
        send_email(title, body + f"\n\nSheet: {sh.url}")
    elif not alerts:
        print("Nothing needs action right now.")
    print("Sheet:", sh.url)

In [ ]:
# Mark an item as replaced
def mark_replaced(item_name, date=None):
    """mark_replaced('Air purifier filter')  -> sets Last Replaced to today and resets alerts."""
    sh, ws, _ = setup(seed=False)
    date = date or dt.date.today().isoformat()
    for r_i, row in enumerate(ws.get_all_values()[1:], start=2):
        if row and row[0].strip().lower() == item_name.strip().lower():
            ws.batch_update([{"range": col_a1(r_i, "last"), "values": [[date]]},
                             {"range": col_a1(r_i, "notified"), "values": [[""]]}],
                            value_input_option="USER_ENTERED")
            print(f"Marked '{item_name}' replaced on {date}")
            return
    print("No item found named:", item_name)

In [ ]:
# Run it
run()